## 1) Criacao do database bronze e mapeamento dos arquivos
Dicionario tabelas_bronze: nome do arquivo CSV -> nome da tabela Bronze.
Bloco abaixo: imports, cria o database bronze, define a pasta do Volume e o dicionario.

In [0]:
#Imports (funcoes do Spark e requests para a API)
from pyspark.sql import functions as f
import requests

#Cria o database da camada Bronze (nao recria se ja existir)
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")


#Pasta do Volume onde estao os 5 CSVs
pasta="/Volumes/workspace/landing/inputs"

#Mapeamento: arquivo CSV -> tabela Bronze
tabelas_bronze = {"movies_info_TMDB_IMDB.csv" : "bronze.tb_movies_info",
                  "movies_financials_IMDB_TMDB.csv" : "bronze.tb_movies_financials",
                  "movies_metrics_IMDB_TMDB.csv" : "bronze.tb_movies_metrics",
                  "credits_and_tags_IMDB_TMDB.csv" : "bronze.tb_credits_and_tags",
                  "movies_reviews.csv" : "bronze.tb_movies_reviews"
                   }

## 2) Leitura dos CSVs e gravacao nas tabelas Bronze
DataFrame df (criado a cada volta do loop) <- CSV do Volume.
Bloco abaixo: para cada arquivo, le o CSV, adiciona ingestion_datetime e grava em Delta (append).

In [0]:
#Para cada par (arquivo, tabela) do dicionario
for arquivo, tabela in tabelas_bronze.items():
    #Le o CSV sem alterar nada (inferSchema desligado: tudo entra como texto, copia fiel)
    df=spark.read.format("csv").option("header","true").option("inferSchema","false").load(f"{pasta}/{arquivo}")

    #Unica coluna nova permitida na Bronze: momento da ingestao
    df = df.withColumn("ingestion_datetime", f.current_timestamp())

    #Grava em Delta com append (cada execucao acumula uma nova carga)
    df.write.format("delta").mode("append").saveAsTable(tabela)
    df.show(5)
    

+------+---------+--------------------+--------------------+-----------------+------------+-------+--------+--------------------+--------------------+--------------------+
|    id|   tconst|               title|      original_title|original_language|release_date|runtime|  status|            overview|             tagline|  ingestion_datetime|
+------+---------+--------------------+--------------------+-----------------+------------+-------+--------+--------------------+--------------------+--------------------+
|293660|tt1431045|            Deadpool|            Deadpool|               en|  2016-02-09|    108|Released|The origin story ...|Witness the begin...|2026-09-20 21:19:...|
|299536|tt4154756|AVENGERS: INFINIT...|Avengers: Infinit...|               en|  04-25-2018|    149|Released|As the Avengers a...|An entire univers...|2026-09-20 21:19:...|
|299534|tt4154796|   Avengers: Endgame|   Avengers: Endgame|               en|  2019-04-24|    181|released|After the devasta...|  Avenge th

## 3) Conferencia
Conta as linhas de cada tabela Bronze gravada.

In [0]:
#Contagem de linhas de cada tabela Bronze
for tabela in tabelas_bronze.values():
    qtd = spark.table(tabela).count()
    print(tabela, qtd)

bronze.tb_movies_info 106930
bronze.tb_movies_financials 106165
bronze.tb_movies_metrics 107364
bronze.tb_credits_and_tags 106320
bronze.tb_movies_reviews 32412


## 4) Extracao da cotacao do dolar via API do Banco Central
DataFrame df_dolar <- API PTAX (Banco Central).
Bloco abaixo: cria os widgets de data, monta a URL, chama a API, cria df_dolar, adiciona ingestion_datetime e grava em bronze.tb_cotacao_dolar.

In [0]:
#Widgets: datas de inicio e fim no formato MM-DD-AAAA
dbutils.widgets.text("data_inicio", "09-12-2026")
dbutils.widgets.text("data_fim", "09-19-2026")

inicio = dbutils.widgets.get("data_inicio")
fim = dbutils.widgets.get("data_fim")
print(inicio, fim)

#Monta a URL do endpoint com as datas dos widgets
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{inicio}'&@dataFinalCotacao='{fim}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

#Chama a API
resposta = requests.get(url)

#Verificacao dos dados da API
if resposta.status_code == 200:
    print("ok")
else :
    print("error")

#Converte a resposta em JSON; os registros ficam na chave "value"
dados = resposta.json()
print(dados)

lista = dados["value"]

#Criacao do Data frame de cotacao do dolar
df_dolar = spark.createDataFrame(lista)
#Adiciona a coluna de controle e grava em Delta (append)
df_dolar = df_dolar.withColumn("ingestion_datetime", f.current_timestamp())
df_dolar.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")
df_dolar.show(5)



09-12-2026 09-19-2026
ok
{'@odata.context': 'https://was-p.bcnet.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata$metadata#_CotacaoDolarPeriodo(cotacaoCompra,dataHoraCotacao)', 'value': [{'cotacaoCompra': 5.169, 'dataHoraCotacao': '2026-09-14 13:10:08.144425'}, {'cotacaoCompra': 5.1484, 'dataHoraCotacao': '2026-09-15 13:09:19.199664'}, {'cotacaoCompra': 5.152, 'dataHoraCotacao': '2026-09-16 13:05:30.35873'}, {'cotacaoCompra': 5.1515, 'dataHoraCotacao': '2026-09-17 13:03:21.858212'}, {'cotacaoCompra': 5.1569, 'dataHoraCotacao': '2026-09-18 13:03:34.742036'}]}
+-------------+--------------------+--------------------+
|cotacaoCompra|     dataHoraCotacao|  ingestion_datetime|
+-------------+--------------------+--------------------+
|        5.169|2026-09-14 13:10:...|2026-09-20 21:19:...|
|       5.1484|2026-09-15 13:09:...|2026-09-20 21:19:...|
|        5.152|2026-09-16 13:05:...|2026-09-20 21:19:...|
|       5.1515|2026-09-17 13:03:...|2026-09-20 21:19:...|
|       5.1569|2026-09-18 13:03

## 5) Limpeza das tabelas (opcional)
Bloco abaixo (comentado): apaga as tabelas Bronze para recomecar do zero. Usar somente em testes, pois apaga todos os dados.

In [0]:
#Descomente para apagar as tabelas Bronze (uso apenas em testes)
#for arquivo,tabela in tabelas_bronze.items():
#spark.sql(f"DROP TABLE IF EXISTS {tabela}")

#spark.sql(f"DROP TABLE IF EXISTS bronze.tb_cotacao_dolar")
